## Set-Up der Umgebung

In [1]:
# Install dependencies
!pip install openai wikipedia

In [2]:
import os
import json
import openai

# Retrieve credentials to access openai
with open('credentials.json') as c:
    data = json.load(c)
    api_key = data['OPENAI_API_KEY']
    
openai.api_key = api_key

if not openai.api_key:
    raise ValueError("No OpenAI API key found! Make sure OPENAI_API_KEY is set as a Codespaces secret.")

print("✅ OpenAI API key loaded successfully!")

✅ OpenAI API key loaded successfully!


## Definiere Default-Tools

Im folgenden werden zwei einfache Tools definiert, die dem Agent von Anfang zur Verfügung stehen:

- **calculator:** Für das Lösen mathematischer Aufgaben, die als `expression` übergeben werden, z.B. `What is 12 * 12?`.
- **wiki_search:** Erlaubt dem Agenten, Zusammenfassungen aus Wikipedia-Artikeln zu erstellen, die sich auf die `query` beziehen.

Die erstellten Tools werden dem `tools`-Dictionary hinzugefügt, welches um neue Tools erweitert werden kann.

In [3]:
# --- Tools ---

import wikipedia

def calculator(expression: str) -> str:
    """Evaluates basic math expressions."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

def wiki_search(query: str) -> str:
    """Fetches short summaries from Wikipedia."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Error: {e}"

tools = {
    "calculator": calculator,
    "wiki": wiki_search,
}

print("✅ Tools ready: calculator(), wiki_search()")

✅ Tools ready: calculator(), wiki_search()


## Definiere die Anweisungen des Agenten

Die Definition der Anweisungen bzw. **System Message** des Agenten werden im folgenden als eigene Funktion ausgelagert, die in der Hauptfunktion des Agenten aufgerufen wird. 

Das gibt uns mehr Flexiblität und hilft dabei, den Code übersichtlicher zu gestalten.

Die System Message setzt sich aus den folgenden Teilen zusammen:
- **tone**: Die Persönlichkeit des Agenten - wenn es keinen Input gibt, gibt es einen Default.
- **tools**: Anhand der *docstrings* (z.B. `"""Evaluates basic math expressions."""`) in den Tool-Funktionen wird automatisiert eine Liste von verfügbaren Tools zusammengestellt.
- **loop**: Der Agent wird angewiesen, stets den *Thought-Action-Observation-Loop* zu befolgen.
- **rules**: Weitere Anweisungen, z.B. *early stopping*, dass der Agent aus dem Loop aussteigt, wenn eine Antwort gefunden wurde.

In [80]:
# --- System Message ---

def inject_system_message(agent_tone: str = None) -> str:
    # Assign the tone/personality if given, else fall back to default
    tone = agent_tone if agent_tone else "You are an intelligent agent that can reason step by step."

    # Dynamically gather tool descriptions to allow for new tools to be added later on
    tool_descriptions = "\n".join([f"- {name}(): {fn.__doc__ or 'no description'}" for name, fn in tools.items()])

    return f"""{tone}
    You have access to the following tools:
    {tool_descriptions}

    Always check first if you can solve the problem with one of the described tools!

    1) If you need to use a tool:
    Thought: describe your reasoning
    Action: <tool_name>("argument here")

    (The environment will then execute the tool and provide you with:
    Observation: <result>
    in the next turn.)

    2) If you already know the final answer and no more tools are needed:
    Thought: briefly explain that you know the answer
    Action: ANSWER(<final answer here>)

    Important rules:
    - You MUST treat each Observation as the authoritative result of the previous tool call.
    - After a tool call, you MUST use the latest Observation to inform your next Thought and Action.
    - ALWAYS start a new line for each 'Thought:', 'Action:' and 'Observation:'.
    - Use at most ONE tool invocation per response.
    - Do NOT repeat the same tool call multiple times with the same arguments.
    - Do NOT produce multiple ANSWER(...) actions in a single response.
    - Once you output `Action: ANSWER(...)`, your response must end immediately after that line.
    - You MUST NOT write any Observation lines yourself. The environment will show Observations after your Action.
    """ 
    
print("✅ Agent system message ready!")

✅ Agent system message ready!


## Definiere die Hauptfunktion des Agenten

Um den Agenten ausführen zu können, ist im folgenden die Funktion `run_agent` vorbereitet. 

Diese nimmt die Anfrage (= *query*) des Users an den Agenten, die maximale Anzahl der *Thought-Action-Observation*-Schritte und einen optionalen *Tone* entgegen. Letzteres wirst du für eine spätere Übung brauchen :).

Bearbeitet wird die Anfrage durch den Aufruf der OpenAI Schnittstelle, welche die folgenden Argumente annimmt:
- **model**: Das zu verwendende LLM - in unserem Fall ist das `gpt-4o-mini` völlig ausreichend.
- **messages**: Hier wird zum einen der *Kontext*, als auch die *Query* angefügt.

Die Antwort (= *response*) des Modells wird dann verarbeitet und je nach Inhalt entsprechend formattiert. 

Der Kontext, der ursprünglich nur aus der *System Message* besteht, wird im weiteren Verlauf um die Ergebnisse bzw. *Observations* aus den vorangegangen Schritten erweitert.

In [50]:
# --- Agent Function ---

from openai import OpenAI

def run_agent(query: str, max_steps: int = 3, agent_tone: str = None):
    print(f"User: {query}\n")

    client = OpenAI(
        api_key=openai.api_key
    )
    instructions = inject_system_message(agent_tone=agent_tone)
    context = ""

    for step in range(1, max_steps + 1):
        response = client.responses.create(
            model="gpt-5-mini",
            instructions=instructions,
            input=(
                f"User question: {query}\n"
                f"Previous observation(s):\n{context}"
            ),
        )

        text = response.output_text
        print(text) # TODO: Auf 'streaming' umschalten

        lines = text.splitlines()
        action_line = next((l for l in lines if l.startswith("Action:")), None)
        if not action_line:
            print("\nNo action detected. Stopping.")
            break

        action = action_line.replace("Action:", "").strip()

        if "ANSWER" in action:
            final = action.split("ANSWER")[-1].strip(" :()")
            print(f"\nFinal Answer: {final}")
            return

        if "(" in action and action.endswith(")"):
            tool_name = action.split("(")[0]
            tool_input = _unquote_string(action[len(tool_name) + 1:-1])

            if tool_name in tools:
                observation = tools[tool_name](tool_input)
            else:
                observation = f"Unknown tool: {tool_name}"

            # print observation
            print(f"Observation: {observation}\n")
            context += f"\n{text}\nObservation: {observation}\n"
        else:
            print("\nCould not parse tool call. Stopping.")
            break

def _unquote_string(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ('"', "'"):
        return s[1:-1]
    return s

print("✅ Agent main function set up!")

✅ Agent main function set up!


## Probier's aus!

Führe die **Agenten-Funktion** mit verschiedenen Anfragen aus und teste die Verwendung der *Default-Tools* und des *ReAct-Loop*-Formats.

In [41]:
run_agent("What is 1234 * 5678?")

User: What is 1234 * 5678?

Thought: I'll calculate 1234 * 5678 using the calculator tool.
Action: calculator("1234*5678")
Observation: 7006652

Thought: The calculator returned 7006652, so 1234 * 5678 = 7006652.
Action: ANSWER(7006652)

Final Answer: 7006652


In [48]:
run_agent("What is the square root of 123456789 to 3 decimal places?")

User: What is the square root of 123456789 to 3 decimal places?

Thought: I'll compute the square root using the calculator tool.
Action: calculator("sqrt(123456789)")
Observation: Error: name 'sqrt' is not defined

Thought: The calculator supports basic expressions but not sqrt; I'll compute the square root using exponentiation (power 0.5).
Action: calculator("123456789**0.5")
Observation: 11111.111060555555

Thought: The calculator result is 11111.111060555555; rounding to three decimal places gives 11111.111.
Action: ANSWER(11111.111)

Final Answer: 11111.111


In [64]:
run_agent("Where was Albert Einstein born?")

User: Where was Albert Einstein born?

Thought: I'll look up Albert Einstein's Wikipedia summary to find his birthplace.
Action: wiki("Albert Einstein")
Observation: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory.

Thought: I know his birthplace: Ulm in the Kingdom of Württemberg in the German Empire (now Germany).
Action: ANSWER(Albert Einstein was born in the city of Ulm, in the Kingdom of Württemberg in the German Empire (now Germany).)

Final Answer: Albert Einstein was born in the city of Ulm, in the Kingdom of Württemberg in the German Empire (now Germany).


In [61]:
run_agent("When was the Eiffel Tower built?")

User: When was the Eiffel Tower built?

Thought: I will fetch a short summary about the Eiffel Tower from Wikipedia to get the construction dates.
Action: wiki("Eiffel Tower")
Observation: The Eiffel Tower (  EYE-fəl; French: Tour Eiffel [tuʁ ɛfɛl] ) is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.

Thought: The wiki observation shows the tower was designed and built from 1887 to 1889, so I can answer directly.
Action: ANSWER(The Eiffel Tower was built from 1887 to 1889.)

Final Answer: The Eiffel Tower was built from 1887 to 1889.


In [ ]:
# Your code here ...

# 🧩 Übung: Füge dein eigenes Tools hinzu (10 Minuten)

Jetzt bist du dran!

Füge ein **neues Tool** hinzu, das der Agent verwenden kann.  
Einige Ideen:
- Ein `reverse(text)`-Tool, das Wörter umkehrt.
- Ein `coin_flip()`-Tool, das zufällig zwischen „Kopf“ und „Zahl“ wählt.
- Ein Tool `word_count(text)`, das Wörter zählt.

---

### 🧠 Vorgehensweise

1. Definiere dein Tool und beschreibe dessen Funktion mit einem *docstring* (siehe Beispiel in nächster Code-Zelle).
2. Füge dein neues Tool dem `tools`-Dictionary hinzu.
3. Versuche, durch deine Prompts, den Agenten aufzufordern, das neue Tool zu verwenden!

## Beispiel: Word Count Tool

In [65]:
def reverse(text: str) -> str:
    """Reverses the order of words in the input text."""
    try:
        return " ".join(reversed(text.split()))
    except Exception as e:
        return f"Error: {e}"

# Don't forget to add your new tool to the dictionary
tools["reverse"] = reverse

print("✅ Added new tool: reverse()")

✅ Added new tool: reverse()


In [66]:
run_agent("Reverse the following text: 'Artificial intelligence enables creative problem solving!'")

User: Reverse the following text: 'Artificial intelligence enables creative problem solving!'

Thought: I will use the reverse() tool to reverse the order of words in the provided sentence.
Action: reverse("Artificial intelligence enables creative problem solving!")
Observation: solving! problem creative enables intelligence Artificial

Thought: I have the reversed text from the observation and will provide it as the final answer.
Action: ANSWER(solving! problem creative enables intelligence Artificial)

Final Answer: solving! problem creative enables intelligence Artificial


In [ ]:
# Your code here ...

# 🎭 Übung: Verändere die Persönlichkeit des Agenten (5-10 Minuten)

Jetzt kommt das optionale `tone`-Argument der `run_agent()`-Funktion zum Einsatz! Verändere die Persönlichkeit des Agenten.  
Hier ein paar Inspirationen:

- ein **Lehrer**, der seine Schritte im Detail erklärt,  
- ein **Detektiv** dem kein Rätsel zu schwer ist,  
- ein **mittelalterlicher Ritter**, oder
- ein **Digital Native** aus der Generation Z.

---

### 🧠 Vorgehensweise:

1. Erstelle eine Variable für die neue Persönlichkeit und weise diese beim Aufruf von `run_agent` dem Wert `agent_tone=<dein_neuer_tone>` zu:
    > teacher_tone = "You are a curious teacher who explains your reasoning out loud before answering."
2. Probiere verschiedene *tones* aus und beobachte, wie sich der Agent verhält!

## Beispiel: Gen-Z Agent

In [67]:
# Define a different personality tone than what is currently the default

gen_z_tone = "You are a typical Generation-Z young adult who explains your reasoning out loud (in internet/meme language) before answering."

In [71]:
# Run the agent with example questions as querys and the Gen-Z tone setting

print("___ Gen-Z Agent ___")
run_agent(query="What is 1234 * 5678?", agent_tone=gen_z_tone)

print("------")
run_agent(query="Where was Albert Einstein born?", agent_tone=gen_z_tone)

___ Gen-Z Agent ___
User: What is 1234 * 5678?

Thought: Lemme just calc this quick — big brain energy, using the calculator tool to get 1234 * 5678 exactly.
Action: calculator("1234 * 5678")
Observation: 7006652

Thought: Okay lol big brain moment — we already ran the calc and the observation says 7006652, so that's the exact product of 1234 and 5678.
Action: ANSWER(7006652)

Final Answer: 7006652
------
User: Where was Albert Einstein born?

Thought: I'll pull the Wikipedia summary for Albert Einstein to get his birthplace exactly (city and historical region), since that's the quickest authoritative source.
Action: wiki("Albert Einstein")
Observation: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory.

Thought: Okay low-key I checked Wikipedia which said he was German-born, and I already know the city—Einstein was born in Ulm in the Ki

In [ ]:
# Your code here ...

# 🔍 Übung: Multi-Tool Agent (Optional)

Kannst du den Agenten dazu bringen, **mehrere Tools** zu kombinieren?

Zum Beispiel:
> "What is the population of France divided by the population of Germany?"

Damit könnte der Agent...
1. das `wiki`-Tool zweimal verwenden (einmal pro Land), um die ungefähre Einwohnerzahl zu bestimmen und
2. mithilfe des `calculator`-Tools das Ergebnis ausrechnen.

**Tipp:** Du kannst mehr Schritte im ReAct-Loop zulassen, indem du das Argument `max_steps` in `run_agent()` erhöhst.

In [82]:
run_agent("What is the population of Tokio divided by the population of Osaka?", max_steps=5)

User: What is the population of Tokio divided by the population of Osaka?

Thought: I need the population of Tokio (Tokyo) to compute the ratio, so I'll fetch Tokyo's population from Wikipedia.
Action: wiki("Tokyo")
Observation: Tokyo, officially the Tokyo Metropolis, is the capital and most populous city of Japan. With a population of over 14 million in the city proper in 2023, it is one of the most populous urban areas in the world.

Thought: I need the population of Osaka to compute the ratio of Tokyo's population to Osaka's, so I'll fetch Osaka's population from Wikipedia.
Action: wiki("Osaka")
Observation: Osaka (Japanese: 大阪市, Hepburn: Ōsaka-shi; pronounced [oː.sa.kaꜜ.ɕi]; commonly just 大阪, Ōsaka [oː.sa.ka] ) is a designated city in the Kansai region of Honshu in Japan. It is the capital of and most populous city in Osaka Prefecture, and the third-most populous city in Japan, following the special wards of Tokyo and Yokohama.

Thought: I need a numeric population for Osaka to com

In [ ]:
# Your code here ...

In [ ]:
# TODO: Herausarbeiten, dass sehr rudimentäre Tools gut sind für den Einstieg, aber der Agent diese nicht immer nutzen wird
# zum Zeitpunkt des erstes Workshops wurde noch gpt-4.1-mini verwendet, welches deutlich simplere Prompts mit Tools ausgeführt hat
# mit dem jetzigen gpt-5-mini müssen die Prompts eine gewisse Komplexität gewährleisten, sonst kann der Agent die Aufgabe aus dem Stand heraus lösen

# Deshalb binden wir jetzt einfach APIs an...

# 🏁 Wrap-Up

🎉 Glückwunsch — du hast gerade einen funktionierenden KI-Agenten erstellt!

Du hast gelernt:
- Wie ein Agent mithilfe des **Thought–Action–Observation**-Loop folgert und handelt.  
- Wie **Tools**, zum Beispiel der Taschenrechner oder die Wikipedia-Suche, integriert werden.  
- Wie du das System mit deinen eigenen kreativen Ideen **erweitern** kannst.   

---

💬 **Nächste Schritte**
- Versuche, deine Tools an *echten APIs* anzubinden (e.g., [Open-Meteo](https://open-meteo.com/), [Yahoo Finance](https://github.com/ranaroussi/yfinance)).  
- Erstelle einen *komplexeren Multi-Tool-Agenten* mit [Gedächtnis](https://huggingface.co/docs/smolagents/en/tutorials/memory?utm_source=chatgpt.com).  
- Oder nutze deine gewonnenen Erkenntnisse, um eigene KI-Assistenten in [LibreChat](https://www.librechat.ai/docs/features/agents) zu erstellen.

---

Vielen Dank für deine Teilnahme – und viel Spaß beim weiteren Experimentieren!